# 임베딩 모델 A/B 테스트 (qwen3-embedding:0.6b vs KURE-v1)

**목적:** 우리 데이터(지자체 복지 정책 574 + PDF 섹션 1,189)와 어르신 말투 질문(v2, 40문항)에 더 잘 맞는 **임베딩 구성**을 고르고, 같은 실험에서 **청킹 방식(A_fixed / B_structure)** 도 비교한다.

| | A_fixed (3,400) | B_structure (3,865) |
|---|---|---|
| qwen3-embedding:0.6b (Ollama, Q8_0, `num_ctx=2048`) | ○ | ○ |
| KURE-v1 (sentence-transformers, fp32, 정규화, 장치는 4-1 환경 기록) | ○ | ○ |

> **비교 대상은 "모델 구조"가 아니라 "실행 구성"이다.** Qwen은 Ollama의 Q8_0 양자화 모델, KURE는 HuggingFace fp32 실행(장치는 `KURE_DEVICE`, 4-1에 기록)이므로 결과는 이 두 구성의 비교다.
> `num_ctx=2048`은 이번 실험 설정이며 Qwen 모델 한계(32K)가 아니다.

**고정 조건**
- 두 모델 모두 **지시문(instruct prefix) 없이** 문서·질문을 그대로 임베딩한다 → 결과는 "지시문 없음 조건"으로 한정해서 해석한다.
- 같은 원본, 같은 질문·정답, k=5, COSINE, 같은 지역 필터, 청크 `text`만 임베딩.
- 모델 비교 = 같은 청킹 안에서 / 청킹 비교 = 같은 모델 안에서 → 항상 하나만 다르게.

**채점 단위:** 청크 Top-5를 **부모 ID**(`parent_id` = 정책 policy_id / PDF section_id)로 채점한다. "서로 다른 정책 5개"를 평가하는 것이 아니다.

**평가 지표**

| 지표 | 뜻 |
|---|---|
| Hit@1 / Hit@3 / Hit@5 | k등 안에 정답이 하나라도 있는 비율 |
| Recall@5 | 정답이 여러 개인 질문에서 5등 안에 찾은 정답 수 ÷ 정답 수 |
| MRR@5 | 첫 정답 순위 점수 평균 (1등=1, 2등=0.5 … 5등 밖=0) |

**선정 기준 (사용자 결정, 실험 전에 정함):** ① **검색 정확도** 우선 — MRR@5(A·B 평균), 보조로 Hit@5·Recall@5 → ② **지역 오류 확인** — 함정 점검 표(금지 정책·타지역 정책 유입), 필터 검증, 지역짝 질문의 차단 청크 유입 → ③ 정확도가 비슷하면(MRR@5 차이 0.02 미만) **속도와 운영 부담**(임베딩·검색 시간, 로컬 실행 여부, 메모리, 의존성). 청킹 방식은 선택한 구성에서 같은 기준.

**지역 필터 전제 (데이터 설계):** 정책 청크의 `sido_name`은 항상 채워져 있고, `sigungu_name`이 비어 있으면 **시도 전체 정책**이다. PDF는 `sido_name`이 비어 있어 전국 지침으로 항상 통과한다. `user_sigungu`가 없는 질문은 그 시도의 모든 시군구 정책을 허용한다. 지역짝 질문의 성능은 **필터 적용 후 검색 성능**이다.

**실행 흐름:** ① 전체 데이터 검증(개수·정답 존재·지역 값) → ② 잘림 검증(청크+질문 40개) 겸 Qwen 문서 벡터 생성 → ③ 두 임베딩 경로 동일성 확인 → ④ smoke(소규모) 적재·필터 검증·평가 → ⑤ 본 실행(보관한 벡터 재사용). 검증 단계가 실패하면 `raise`로 멈춘다.

## 1. 설정
- `CHUNK_SETS`: A·B 방식마다 **정책 파일 + PDF 파일** 2개
- `EXPECTED_COUNTS` / `EXPECTED_SIGUNGU_NULL`: 사전 확인한 청크 수·시도전체 정책 청크 수 → 다르면 중단
- `SMOKE = True`: 소규모 사전 점검(질문 6개 + 관련 청크 + 무작위 200개). 본 실행은 `False`
- `REGION_PREMISE_CONFIRMED`: 2-1 셀의 시도전체 정책 목록을 **눈으로 확인한 뒤** `True`로 바꿔야 다음 단계가 진행됨
- `KURE_REV`, `QWEN_TOK_REV`: 재현성을 위한 HuggingFace 커밋 해시 고정
- `DATA_ROOT`: 청크 폴더 위치. Runpod에서는 업로드한 위치로 바꾼다. `KURE_DEVICE`는 GPU가 있으면 자동으로 `cuda`

In [1]:
import time, uuid
import torch

DATA_ROOT = "../../data/rag/chunks"          # Runpod 등 다른 환경이면 청크 폴더 위치만 바꾼다
CHUNK_SETS = {
    "A_fixed": [f"{DATA_ROOT}/A_fixed/policy_chunks.jsonl", f"{DATA_ROOT}/A_fixed/guideline_chunks.jsonl"],
    "B_structure": [f"{DATA_ROOT}/B_structure/policy_chunks.jsonl", f"{DATA_ROOT}/B_structure/guideline_chunks.jsonl"],
}
KURE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # GPU가 있으면 자동으로 사용 (환경 기록에 남음)
EXPECTED_COUNTS = {"A_fixed": 3400, "B_structure": 3865}          # 정책 + PDF 청크 수
EXPECTED_SIGUNGU_NULL = {"A_fixed": 66, "B_structure": 70}       # 시군구가 비어 있는 정책 청크 수(시도 전체 정책)
TEST_PATH = "./data/test_questions_v3.jsonl"

K = 5                                    # Hit@5, Recall@5, MRR@5
USE_REGION_FILTER = True                 # 지역 질문에 지역 필터 (모든 모델 똑같이)
INCLUDE_TYPES = ("policy", "guideline")  # 시설 청크 제외
NUM_CTX = 2048                           # Ollama 실행 컨텍스트(실험 설정, 모델 한계 아님)

KURE_REV = "8b418a58414668e75532ed045c22d9ca018ae2b2"      # nlpai-lab/KURE-v1 (2026-11-10 확인)
QWEN_TOK_REV = "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3"  # Qwen/Qwen3-Embedding-0.6B 토크나이저

SMOKE = False                           # True: 소규모 사전 점검 / False: 본 실행
SMOKE_QIDS = ["V01", "V13", "V19", "V25", "V27", "V37"]   # 지역정책 서울·전남, 지역짝, 시도전체, PDF, 함정
SMOKE_BLOCK_FROM = {"V01": ["V02"], "V13": ["V07"], "V19": ["V20"]}  # 이 질문에서 차단돼야 할 "다른 질문의 정답"
SMOKE_RANDOM = 200                       # 무작위로 섞을 청크 수
SEED = 42

REGION_PREMISE_CONFIRMED = True        # 2-1 셀 목록 확인 후 True

RUN_ID = time.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
RESULT_DIR = f"./results/{RUN_ID}"
print("RUN_ID:", RUN_ID, "| SMOKE:", SMOKE, "| KURE_DEVICE:", KURE_DEVICE,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

RUN_ID: 20261110_062652_1d153850 | SMOKE: False | KURE_DEVICE: cuda | GPU: NVIDIA GeForce RTX 4090


## 2. 청크 읽기
- 강사님 `JSONLoader` 사용. `metadata` 안의 지역·출처 정보를 **펼쳐서** 읽는다
- 방식마다 정책 파일과 PDF 파일을 읽어서 합친다
- 개수가 사전 확인값과 다르면 **중단**

In [2]:
from collections import Counter
from langchain_community.document_loaders import JSONLoader

def metadata_func(record: dict, metadata: dict) -> dict:
    for key, value in record.items():
        if key == "text":
            continue
        if key == "metadata" and isinstance(value, dict):
            metadata.update(value)      # metadata 안의 정보를 펼치기
        else:
            metadata[key] = value
    return metadata

def load_documents(paths):
    docs = []
    for path in paths:                  # 정책 파일 + PDF 파일
        loader = JSONLoader(
            file_path=path,
            jq_schema=".",
            content_key="text",
            json_lines=True,
            metadata_func=metadata_func,
        )
        docs.extend(loader.load())
    docs = [d for d in docs if d.metadata.get("record_type", "policy") in INCLUDE_TYPES]
    for d in docs:
        if not d.metadata.get("parent_id"):
            raise RuntimeError(f"parent_id 없는 청크: {d.metadata.get('chunk_id')}")
    return docs

chunk_data = {}
for chunk_name, paths in CHUNK_SETS.items():
    docs = load_documents(paths)
    chunk_data[chunk_name] = docs
    print(f"[{chunk_name}] 청크 수: {len(docs)} (기대 {EXPECTED_COUNTS[chunk_name]})",
          Counter(d.metadata.get("record_type") for d in docs))
    if len(docs) != EXPECTED_COUNTS[chunk_name]:
        raise RuntimeError(f"[{chunk_name}] 청크 수가 사전 확인값과 다름 → 파일 경로·버전 확인")

/tmp/ipykernel_2242/2602114779.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader


[A_fixed] 청크 수: 3400 (기대 3400) Counter({'guideline': 2738, 'policy': 662})
[B_structure] 청크 수: 3865 (기대 3865) Counter({'guideline': 3190, 'policy': 675})


### 2-1. 지역 값 확인 (필터 전제 검증)
- 정책 청크에 `sido_name`이 비어 있으면 **중단** (누락 데이터가 필터를 통과하면 안 됨)
- `sigungu_name`이 비어 있는 정책 청크 수가 사전 확인값과 다르면 **중단**
- 아래 목록이 정말 "시도 전체 정책"인지는 자동으로 알 수 없으므로 **사람이 보고** 설정 셀의 `REGION_PREMISE_CONFIRMED`를 `True`로 바꾼다
- 2026-11-10 확인 결과 기록: 53개 대부분 시도 단위 정책. **예외** ① `수원형 통합돌봄 수원새빛돌봄`(경기도, 시군구 null) — 실제로는 수원시 정책이지만 필터가 정상이어도 경기도 모든 시군구 질문에 통과한다(데이터 전제의 예외, 결과 해석 시 명시) ② `전남광주통합특별시` 시도명 2건 — 지역 필터가 있는 질문에서는 차단되지만 지역 미지정 질문에서는 통과할 수 있다

In [3]:
import pandas as pd

for chunk_name, docs in chunk_data.items():
    policies = [d for d in docs if d.metadata.get("record_type") == "policy"]
    sido_null = [d for d in policies if d.metadata.get("sido_name") is None]
    if sido_null:
        raise RuntimeError(f"[{chunk_name}] sido_name 비어 있는 정책 청크 {len(sido_null)}개 → 필터 전제 위반")
    sgg_null = [d for d in policies if d.metadata.get("sigungu_name") is None]
    if len(sgg_null) != EXPECTED_SIGUNGU_NULL[chunk_name]:
        raise RuntimeError(f"[{chunk_name}] sigungu_name 비어 있는 정책 청크 {len(sgg_null)}개 "
                           f"(기대 {EXPECTED_SIGUNGU_NULL[chunk_name]}) → 데이터 변경 여부 확인")
    table = (pd.DataFrame([{"parent_id": d.metadata["parent_id"], "sido": d.metadata["sido_name"],
                            "title": d.metadata.get("title")} for d in sgg_null])
             .drop_duplicates("parent_id").sort_values(["sido", "title"]))
    print(f"\n[{chunk_name}] 시군구 비어 있는 정책 청크 {len(sgg_null)}개 → 정책 {len(table)}개 (아래가 전부 '시도 전체' 정책인지 확인)")
    with pd.option_context("display.max_rows", 200, "display.max_colwidth", 60):
        display(table.reset_index(drop=True))

if not REGION_PREMISE_CONFIRMED:
    raise RuntimeError("위 목록을 확인한 뒤 설정 셀에서 REGION_PREMISE_CONFIRMED = True 로 바꾸고 다시 실행하세요.")
print("지역 값 전제 확인 완료")


[A_fixed] 시군구 비어 있는 정책 청크 66개 → 정책 53개 (아래가 전부 '시도 전체' 정책인지 확인)


,parent_id,sido,title
0,local:WLF00000688,경기도,결식아동 급식지원
1,local:WLF00005029,경기도,경기남부청소년자립지원관
2,local:WLF00005270,경기도,경기도 소규모 노후주택 집수리 지원사업
3,local:WLF00003575,경기도,경기도 의약품 안전사용 환경조성 사업
4,local:WLF00003903,경기도,경기도형 긴급복지 지원사업
5,local:WLF00005030,경기도,경기북부청소년자립지원관
6,local:WLF00003847,경기도,국가유공자 생활보조보훈수당 지원
7,local:WLF00003678,경기도,기초생활수급 노인 월동난방비 지원
8,local:WLF00000690,경기도,노숙인 보호 및 지원
9,local:WLF00004070,경기도,도비 장애수당



[B_structure] 시군구 비어 있는 정책 청크 70개 → 정책 53개 (아래가 전부 '시도 전체' 정책인지 확인)


,parent_id,sido,title
0,local:WLF00000688,경기도,결식아동 급식지원
1,local:WLF00005029,경기도,경기남부청소년자립지원관
2,local:WLF00005270,경기도,경기도 소규모 노후주택 집수리 지원사업
3,local:WLF00003575,경기도,경기도 의약품 안전사용 환경조성 사업
4,local:WLF00003903,경기도,경기도형 긴급복지 지원사업
5,local:WLF00005030,경기도,경기북부청소년자립지원관
6,local:WLF00003847,경기도,국가유공자 생활보조보훈수당 지원
7,local:WLF00003678,경기도,기초생활수급 노인 월동난방비 지원
8,local:WLF00000690,경기도,노숙인 보호 및 지원
9,local:WLF00004070,경기도,도비 장애수당


지역 값 전제 확인 완료


## 3. 테스트 질문 + 채점할 수 있는 질문만 고르기
- 질문 세트 v2 (40개): 지역정책 18 · 지역짝 6 · 시도전체 2 · PDF근거 10 · 함정 4
- 함정 질문 4개는 Hit/Recall/MRR에서 제외하고 **8번 함정 점검 표**에서 따로 본다
- **정답 ID가 하나라도 데이터에 없는 질문은 제외** (`all()`): 모델 탓이 아닌데 틀린 걸로 세어지는 걸 막는다
- 두 청킹 세트의 채점 질문이 서로 다르거나 36개가 아니면 **중단**

In [4]:
import json

with open(TEST_PATH, encoding="utf-8") as f:
    all_questions = [json.loads(line) for line in f if line.strip()]
q_by_id = {q["qid"]: q for q in all_questions}
trap_questions = [q for q in all_questions if q["q_type"].startswith("함정")]
scoring_pool = [q for q in all_questions if not q["q_type"].startswith("함정")]
print(f"전체 {len(all_questions)}개 = 채점 후보 {len(scoring_pool)} + 함정 {len(trap_questions)}")

def select_questions(docs):
    parent_ids = {d.metadata["parent_id"] for d in docs}
    questions, skipped = [], []
    for q in scoring_pool:
        if q["expected_ids"] and all(pid in parent_ids for pid in q["expected_ids"]):
            questions.append(q)
        else:
            skipped.append(q)
    return questions, skipped

chunk_questions = {}
for chunk_name, docs in chunk_data.items():
    questions, skipped = select_questions(docs)
    chunk_questions[chunk_name] = questions
    print(f"[{chunk_name}] 채점할 질문: {len(questions)}개 / 정답이 데이터에 없어 제외: {len(skipped)}개")
    for q in skipped:
        print("   제외:", q["qid"], q["question"][:25], "| 정답:", q.get("expected_title"))

qid_sets = {name: {q["qid"] for q in qs} for name, qs in chunk_questions.items()}
if len({frozenset(s) for s in qid_sets.values()}) != 1:
    raise RuntimeError(f"청킹 세트별 채점 질문이 다름: {qid_sets}")
SCORING_QIDS = sorted(next(iter(qid_sets.values())))
if len(SCORING_QIDS) != 113:
    raise RuntimeError(f"채점 질문이 113개가 아님: {len(SCORING_QIDS)}")
print("두 세트 모두 같은 113개 질문으로 채점")

전체 122개 = 채점 후보 113 + 함정 9
[A_fixed] 채점할 질문: 113개 / 정답이 데이터에 없어 제외: 0개
[B_structure] 채점할 질문: 113개 / 정답이 데이터에 없어 제외: 0개
두 세트 모두 같은 113개 질문으로 채점


## 4. 비교할 구성
| 이름 | 구성 | 비고 |
|---|---|---|
| qwen3-embedding:0.6b | Ollama, Q8_0, `num_ctx=2048`, 지시문 없음 | 수업 모델, 기준 |
| KURE-v1 | `nlpai-lab/KURE-v1` revision 고정, fp32, L2 정규화, batch 16, 장치는 `KURE_DEVICE`(GPU 있으면 cuda) | bge-m3를 한국어로 추가 학습한 무료 모델 |

- `OllamaEmbeddings`는 Ollama `/api/embed`를 쓰며 정규화된 벡터를 돌려준다. Qdrant 거리는 COSINE으로 통일.
- `trust_remote_code`는 쓰지 않는다. 첫 실행 시 KURE-v1 약 2.2GB 다운로드. 메모리 부족 시 `batch_size` 8로.

In [5]:
from langchain_ollama import OllamaEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

models = {
    "qwen3-embedding:0.6b": OllamaEmbeddings(model="qwen3-embedding:0.6b", num_ctx=NUM_CTX),
    "KURE-v1": HuggingFaceEmbeddings(
        model_name="nlpai-lab/KURE-v1",
        model_kwargs={"device": KURE_DEVICE, "revision": KURE_REV},
        encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    ),
}

# bge-m3 (이번 비교에서 제외) : models["bge-m3"] = OllamaEmbeddings(model="bge-m3", num_ctx=NUM_CTX)
# OpenAI (유료) / .env에 OPENAI_API_KEY 필요
# from dotenv import load_dotenv; load_dotenv()
# from langchain_openai import OpenAIEmbeddings
# models["text-embedding-3-small"] = OpenAIEmbeddings(model="text-embedding-3-small")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### 4-1. 환경 기록 (`run_meta.json`)
결과를 나중에 해석·재현할 수 있도록 패키지 버전, Ollama 모델 digest·양자화, KURE revision·dtype, CPU/GPU 사용 여부, 입력 파일 해시를 저장한다. 워밍업 호출도 여기서 한다(모델 로딩 시간을 지표에서 분리).

In [6]:
import hashlib, platform, sys, json, urllib.request
import importlib.metadata as im
import torch

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def ollama_get(path):
    with urllib.request.urlopen("http://localhost:11434" + path, timeout=30) as r:
        return json.loads(r.read().decode("utf-8"))

# 워밍업 (Ollama 모델 로딩 포함) + 벡터 길이
dims = {}
for name, m in models.items():
    t0 = time.time()
    dims[name] = len(m.embed_query("준비"))
    print(f"{name}: dim={dims[name]} 워밍업 {time.time()-t0:.1f}초")

qwen_tag = next(m for m in ollama_get("/api/tags")["models"] if m["name"] == "qwen3-embedding:0.6b")
ps = ollama_get("/api/ps").get("models", [])
kure_st = getattr(models["KURE-v1"], "client", None) or models["KURE-v1"]._client   # SentenceTransformer (langchain-huggingface 버전에 따라 이름이 다름)

run_meta = {
    "run_id": RUN_ID, "mode": "smoke" if SMOKE else "full",
    "k": K, "use_region_filter": USE_REGION_FILTER, "num_ctx": NUM_CTX, "instruction_prefix": None,
    "python": sys.version.split()[0], "platform": platform.platform(), "cpu": platform.processor(),
    "packages": {p: im.version(p) for p in ["langchain-ollama", "langchain-huggingface", "langchain-qdrant",
                                            "sentence-transformers", "torch", "transformers", "qdrant-client", "ollama"]},
    "torch_cuda_build": torch.version.cuda, "torch_cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "ollama": {"server_version": ollama_get("/api/version")["version"],
               "model": qwen_tag["name"], "digest": qwen_tag["digest"],
               "quantization_level": qwen_tag["details"].get("quantization_level"),
               "parameter_size": qwen_tag["details"].get("parameter_size"),
               "ps_after_warmup": [{"name": p["name"], "size_vram": p.get("size_vram")} for p in ps]},
    "kure": {"revision": KURE_REV, "device": str(kure_st.device),
             "dtype": str(next(kure_st.parameters()).dtype), "max_seq_length": kure_st.max_seq_length},
    "qwen_tokenizer_revision": QWEN_TOK_REV,
    "dims": dims,
    "input_files": {p: sha256(p) for paths in CHUNK_SETS.values() for p in paths} | {TEST_PATH: sha256(TEST_PATH)},
    "chunk_counts": {n: len(d) for n, d in chunk_data.items()},
    "scoring_qids": SCORING_QIDS, "trap_qids": [q["qid"] for q in trap_questions],
}
print(json.dumps({k: v for k, v in run_meta.items() if k not in ("input_files", "scoring_qids")}, ensure_ascii=False, indent=1))

qwen3-embedding:0.6b: dim=1024 워밍업 0.0초
KURE-v1: dim=1024 워밍업 0.9초
{
 "run_id": "20261110_062652_1d153850",
 "mode": "full",
 "k": 5,
 "use_region_filter": true,
 "num_ctx": 2048,
 "instruction_prefix": null,
 "python": "3.12.3",
 "platform": "Linux-6.8.0-138-generic-x86_64-with-glibc2.39",
 "cpu": "x86_64",
 "packages": {
  "langchain-ollama": "1.1.0",
  "langchain-huggingface": "1.2.3",
  "langchain-qdrant": "1.1.0",
  "sentence-transformers": "6.2.0",
  "torch": "2.8.0+cu128",
  "transformers": "5.22.2",
  "qdrant-client": "1.20.0",
  "ollama": "0.6.3"
 },
 "torch_cuda_build": "12.8",
 "torch_cuda_available": true,
 "gpu_name": "NVIDIA GeForce RTX 4090",
 "ollama": {
  "server_version": "0.40.16",
  "model": "qwen3-embedding:0.6b",
  "digest": "ac6da0dfba84a81fdbfbaf330198c33cd77c4cdfc53e8bc50eb581914a15621d",
  "quantization_level": "Q8_0",
  "parameter_size": "595.78M",
  "ps_after_warmup": [
   {
    "name": "qwen3-embedding:0.6b",
    "size_vram": 985588039
   }
  ]
 },
 "kure":

### 4-2. 잘림(truncation) 검증 + Qwen 문서 벡터 생성 (한 번에)
- 대상 = 두 모델에 **실제로 전달되는 문자열 전부**: 모든 청크 `text` + 질문 40개 원문
- Qwen: `ollama.embed(..., truncate=False, options={"num_ctx": NUM_CTX})` → 컨텍스트 초과 입력이 있으면 Ollama가 오류를 냄 (그 자체가 검증). 설치된 `OllamaEmbeddings.embed_documents`도 내부에서 같은 `Client.embed(model, texts, options={num_ctx,...})`를 호출하므로, **여기서 만든 청크 벡터를 평가에 재사용**한다(같은 계산을 반복하지 않기 위해; Codex AB-6 의견 반영). 4-3에서 두 경로의 벡터가 같은지 확인한다
- KURE: LangChain 래퍼가 문서의 **줄바꿈을 공백으로 바꾼 뒤** 임베딩하므로, 토큰 수도 그 문자열로 센다. 벡터는 본 실행 때 래퍼 경로로 한 번 만들어 같은 커널 안에서 재사용
- 토크나이저 계수: `truncation=False, add_special_tokens=True`. Qwen 최대 ≥ `NUM_CTX` 또는 KURE 최대 > `max_seq_length`면 **중단**
- 재사용 벡터는 커널 메모리(`doc_vectors`)와 **디스크 캐시** `results/cache/*.npz`에 둔다. 캐시 키 = 모델 digest(또는 KURE revision) + `num_ctx` + 청크 파일 sha256 + 청크 수. 키가 같은 캐시가 있으면 잘림 검증 기록과 함께 불러와 임베딩을 건너뛴다(Codex AB-6: "동일 입력·설정의 검증 성공 기록이 있으면 재검증 생략 가능"). 키가 다르면(데이터·모델·설정 변경) 다시 만든다
- `results/cache/`는 커밋 대상이 아니다(커밋 범위 승인 때 `.gitignore` 처리)

In [7]:
import os
import ollama
import numpy as np
from transformers import AutoTokenizer

QWEN = "qwen3-embedding:0.6b"
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", revision=QWEN_TOK_REV)
kure_tok = kure_st.tokenizer
ollama_client = ollama.Client()

def max_tokens(tokenizer, texts):
    enc = tokenizer(texts, truncation=False, add_special_tokens=True, padding=False)
    return max(len(ids) for ids in enc["input_ids"])

def check_vectors(vecs, n, dim, label):
    # 형태·유한값·양수 노름 검사 (메모리 재사용·캐시 저장 전 공통)
    arr = np.asarray(vecs, dtype=np.float32)
    if arr.shape != (n, dim):
        raise RuntimeError(f"[{label}] 벡터 형태 {arr.shape} != {(n, dim)}")
    if not np.isfinite(arr).all():
        raise RuntimeError(f"[{label}] 비유한값 벡터 포함")
    if (np.linalg.norm(arr, axis=1) == 0).any():
        raise RuntimeError(f"[{label}] 0 벡터 포함")
    return arr

def record_failure(kind, info):
    # 예외로 중단되기 전에 디스크에 남긴다 (RESULT_DIR는 7번에서 생기므로 results/ 바로 아래)
    os.makedirs("./results", exist_ok=True)
    path = f"./results/failures_{RUN_ID}.json"
    data = json.load(open(path, encoding="utf-8")) if os.path.exists(path) else {"run_id": RUN_ID, "failures": []}
    data["failures"].append({"kind": kind, "at": time.strftime("%Y-%m-%d %H:%M:%S"), **info})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)
    run_meta.setdefault("failures", []).append(data["failures"][-1])
    print("실패 기록 저장:", path)

def ollama_embed_all(texts, batch=64, ids=None):
    vecs, prompt_eval = [], 0
    for i in range(0, len(texts), batch):
        try:
            r = ollama_client.embed(model=QWEN, input=texts[i:i+batch], truncate=False, options={"num_ctx": NUM_CTX})
        except Exception as e:                                   # 실패 배치 범위·ID·오류 본문을 디스크에 남기고 중단
            record_failure("ollama_embed", {"batch_start": i, "batch_end": i + len(texts[i:i+batch]),
                                            "ids": (ids or [])[i:i+batch], "error_type": type(e).__name__, "error": str(e)[:500]})
            raise
        if len(r["embeddings"]) != len(texts[i:i+batch]):
            record_failure("ollama_count_mismatch", {"batch_start": i, "returned": len(r["embeddings"]),
                                                     "expected": len(texts[i:i+batch]), "ids": (ids or [])[i:i+batch]})
            raise RuntimeError(f"Ollama 반환 개수 불일치: {len(r['embeddings'])} != {len(texts[i:i+batch])} (batch {i})")
        vecs.extend(r["embeddings"]); prompt_eval += r.get("prompt_eval_count") or 0
    return vecs, prompt_eval

CACHE_DIR = "./results/cache"
os.makedirs(CACHE_DIR, exist_ok=True)
MODEL_CACHE_TAG = {QWEN: {"digest": qwen_tag["digest"], "num_ctx": NUM_CTX, "truncate": False},
                   "KURE-v1": {"revision": KURE_REV, "normalize": True, "device": KURE_DEVICE}}

def cache_path(model_name, chunk_name):
    key = {"model": model_name, "tag": MODEL_CACHE_TAG[model_name],
           "files": [run_meta["input_files"][p] for p in CHUNK_SETS[chunk_name]], "n": len(chunk_data[chunk_name])}
    h = hashlib.sha256(json.dumps(key, sort_keys=True).encode()).hexdigest()[:16]
    return f"{CACHE_DIR}/{model_name.replace(':', '_')}__{chunk_name}__{h}.npz"

# 캐시 생성 당시 환경 (사이드카 .meta.json에 저장, 로드 시 현재 환경과 완전히 같아야 사용)
CACHE_ENV = {
    QWEN: {"ollama_server": run_meta["ollama"]["server_version"], "digest": qwen_tag["digest"], "num_ctx": NUM_CTX,
           "truncate": False, "langchain_ollama": run_meta["packages"]["langchain-ollama"], "ollama_py": run_meta["packages"]["ollama"],
           "preprocess": "none"},
    "KURE-v1": {"revision": KURE_REV, "dtype": run_meta["kure"]["dtype"], "max_seq_length": run_meta["kure"]["max_seq_length"],
                "device": run_meta["kure"]["device"], "normalize": True, "batch_size": 16, "preprocess": "newline_to_space",
                "sentence_transformers": run_meta["packages"]["sentence-transformers"], "torch": run_meta["packages"]["torch"],
                "transformers": run_meta["packages"]["transformers"], "langchain_huggingface": run_meta["packages"]["langchain-huggingface"]},
}
PASS_FLAG = {QWEN: "ollama_truncate_false_passed", "KURE-v1": "langchain_embed_documents"}

def save_vectors(path, docs, vecs, embed_sec, source, model_name, extra=None):
    ids = [d.metadata["chunk_id"] for d in docs]
    arr = check_vectors(vecs, len(ids), dims[model_name], f"save {os.path.basename(path)}")
    if len(set(ids)) != len(ids):
        raise RuntimeError("chunk_id 중복 → 캐시 저장 불가")
    tmp = path + ".tmp.npz"
    np.savez(tmp, vectors=arr, chunk_ids=np.array(ids))
    os.replace(tmp, path)                                        # 완성 후 교체 (부분 파일 방지)
    side = {"env": CACHE_ENV[model_name], "model": model_name, "embed_sec": embed_sec, "source": source,
            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"), "n": len(ids), "dim": int(arr.shape[1]), "extra": extra or {}}
    tmp_side = path + ".meta.json.tmp"
    with open(tmp_side, "w", encoding="utf-8") as f:
        json.dump(side, f, ensure_ascii=False, indent=1)
    os.replace(tmp_side, path + ".meta.json")

def load_vectors(path, docs, model_name):
    if not (os.path.exists(path) and os.path.exists(path + ".meta.json")):
        return None
    try:                                                         # 손상된 JSON/NPZ·필수 키 누락 → 사용 안 함 (중단 아님)
        with open(path + ".meta.json", encoding="utf-8") as f:
            side = json.load(f)
        z = np.load(path, allow_pickle=False)
        arr, ids = z["vectors"], list(z["chunk_ids"])
        for k in ("env", "embed_sec", "source", "saved_at", "extra"):
            side[k]
    except Exception as e:
        print(f"   캐시 사용 안 함 {os.path.basename(path)}: 읽기/구조 오류 {type(e).__name__}: {str(e)[:120]}")
        return None
    reasons = []
    if side["env"] != CACHE_ENV[model_name]:
        reasons.append(f"환경 불일치 {side['env']} != {CACHE_ENV[model_name]}")
    if not side["extra"].get(PASS_FLAG[model_name]):
        reasons.append(f"검증 기록 없음({PASS_FLAG[model_name]})")
    want = [d.metadata["chunk_id"] for d in docs]
    if ids != want:
        reasons.append("chunk_id 순서/개수 불일치")
    if len(set(ids)) != len(ids):
        reasons.append("chunk_id 중복")
    if arr.shape != (len(want), dims[model_name]):
        reasons.append(f"벡터 형태 {arr.shape} != {(len(want), dims[model_name])}")
    elif not np.isfinite(arr).all() or (np.linalg.norm(arr, axis=1) == 0).any():
        reasons.append("비유한값 또는 0 벡터 포함")
    if reasons:
        print(f"   캐시 사용 안 함 {os.path.basename(path)}: " + " / ".join(reasons))
        return None
    return {"by_chunk_id": dict(zip(ids, arr.tolist())), "embed_sec": side["embed_sec"],
            "source": f"디스크 캐시({os.path.basename(path)}, {side['saved_at']}) ← {side['source']}", "extra": side.get("extra", {})}

doc_vectors = {name: {} for name in CHUNK_SETS}      # chunk_set → model → {"by_chunk_id": {...}, "embed_sec": float, "source": str}
truncation_check = {}
question_texts = [q["question"] for q in all_questions]

# 질문 40개 검증 (문서 캐시와 무관하게 항상 실행: 질문 파일이 바뀌어도 서버 검증이 생략되지 않도록)
t0 = time.time()
_, pe_q = ollama_embed_all(question_texts, ids=[q["qid"] for q in all_questions])
run_meta["question_truncate_check"] = {
    "n_questions": len(question_texts), "ollama_truncate_false_passed": True, "ollama_prompt_eval_count": pe_q,
    "qwen_max_tokens": max_tokens(qwen_tok, question_texts),
    "kure_max_tokens": max_tokens(kure_tok, [t.replace("\n", " ") for t in question_texts]), "sec": round(time.time() - t0, 1)}
print("질문 40개 Ollama truncate=False 통과 |", run_meta["question_truncate_check"])

for chunk_name, docs in chunk_data.items():
    chunk_texts = [d.page_content for d in docs]
    texts = chunk_texts + question_texts
    kure_texts = [t.replace("\n", " ") for t in texts]           # HuggingFaceEmbeddings 전처리와 동일
    q_max, k_max = max_tokens(qwen_tok, texts), max_tokens(kure_tok, kure_texts)
    if q_max >= NUM_CTX:
        raise RuntimeError(f"[{chunk_name}] Qwen 토큰 {q_max} ≥ num_ctx {NUM_CTX} → 설정 재검토")
    if k_max > kure_st.max_seq_length:
        raise RuntimeError(f"[{chunk_name}] KURE 토큰 {k_max} > max_seq_length {kure_st.max_seq_length}")

    path = cache_path(QWEN, chunk_name)
    cached = load_vectors(path, docs, QWEN)
    if cached is not None:                                        # 같은 모델·환경·데이터의 검증 성공 기록 → 문서 재검증 생략
        doc_vectors[chunk_name][QWEN] = cached
        truncation_check[chunk_name] = cached["extra"] | {"qwen_max_tokens": q_max, "kure_max_tokens": k_max, "from_cache": True}
        print(f"[{chunk_name}] 캐시 로드: {os.path.basename(path)} | Qwen 최대 {q_max} / KURE 최대 {k_max} 토큰 | "
              f"이전 Ollama truncate=False 통과 기록 사용 (문서 임베딩 {cached['embed_sec']}초)")
        continue

    t0 = time.time()
    vecs, pe_docs = ollama_embed_all(chunk_texts, ids=[d.metadata["chunk_id"] for d in docs])   # 청크: 검증 + 벡터 보관
    embed_sec = round(time.time() - t0, 1)
    extra = {"n_chunks": len(chunk_texts), "ollama_prompt_eval_count": pe_docs, "qwen_doc_embed_sec": embed_sec,
             "ollama_truncate_false_passed": True}
    source = "4-2 잘림검증에서 생성(truncate=False)"
    check_vectors(vecs, len(docs), dims[QWEN], f"{chunk_name}/{QWEN}")
    save_vectors(path, docs, vecs, embed_sec, source, QWEN, extra)
    doc_vectors[chunk_name][QWEN] = {"by_chunk_id": {d.metadata["chunk_id"]: v for d, v in zip(docs, vecs)},
                                     "embed_sec": embed_sec, "source": source}
    truncation_check[chunk_name] = extra | {"qwen_max_tokens": q_max, "kure_max_tokens": k_max, "from_cache": False}
    print(f"[{chunk_name}] 텍스트 {len(texts)}개 | Qwen 최대 {q_max} 토큰 (num_ctx {NUM_CTX}) | "
          f"KURE 최대 {k_max} 토큰 (max_seq_length {kure_st.max_seq_length}) | Ollama truncate=False 통과 | "
          f"Qwen 문서 임베딩 {embed_sec:.0f}초 → 캐시 저장 {os.path.basename(path)}")
run_meta["truncation_check"] = truncation_check
print("잘림 없음 확인, Qwen 문서 벡터 보관")

질문 40개 Ollama truncate=False 통과 | {'n_questions': 122, 'ollama_truncate_false_passed': True, 'ollama_prompt_eval_count': 3249, 'qwen_max_tokens': 43, 'kure_max_tokens': 34, 'sec': 2.8}
[A_fixed] 캐시 로드: qwen3-embedding_0.6b__A_fixed__e2210d508ab3baea.npz | Qwen 최대 731 / KURE 최대 606 토큰 | 이전 Ollama truncate=False 통과 기록 사용 (문서 임베딩 34.3초)
[B_structure] 캐시 로드: qwen3-embedding_0.6b__B_structure__a3d7c996e1721ee8.npz | Qwen 최대 727 / KURE 최대 545 토큰 | 이전 Ollama truncate=False 통과 기록 사용 (문서 임베딩 36.9초)
잘림 없음 확인, Qwen 문서 벡터 보관


### 4-3. 두 경로 표본 유사성 확인 (직접 `ollama.embed` vs LangChain `embed_documents`)
대표 청크 20개(앞 10개 + 가장 긴 10개)를 LangChain 경로로 다시 임베딩해 4-2 벡터와 코사인 유사도를 비교한다. 최솟값이 0.99 미만이면 **중단**(재사용 불가). 이것은 경로 이상을 찾는 **표본 검사**이며, 검색 순위가 완전히 같다는 증명은 아니다(동점 순서까지는 보장하지 않음).

In [8]:
import numpy as np

def cosine(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

equiv = {}
for chunk_name, docs in chunk_data.items():
    longest = sorted(docs, key=lambda d: len(d.page_content), reverse=True)[:10]
    sample = docs[:10] + longest
    lc_vecs = models[QWEN].embed_documents([d.page_content for d in sample])
    cached_vecs = [doc_vectors[chunk_name][QWEN]["by_chunk_id"][d.metadata["chunk_id"]] for d in sample]
    check_vectors(lc_vecs, len(sample), dims[QWEN], f"{chunk_name} LangChain 경로")      # 형태·유한값·양수 노름
    check_vectors(cached_vecs, len(sample), dims[QWEN], f"{chunk_name} 4-2 경로")
    sims = [cosine(a, b) for a, b in zip(lc_vecs, cached_vecs)]
    if not np.isfinite(sims).all():
        raise RuntimeError(f"[{chunk_name}] 유사도에 비유한값 포함")
    equiv[chunk_name] = {"n": len(sample), "min_cos": round(min(sims), 6), "mean_cos": round(float(np.mean(sims)), 6)}
    print(f"[{chunk_name}] 두 경로 코사인 유사도 최소 {min(sims):.6f} / 평균 {np.mean(sims):.6f}")
    if min(sims) < 0.99:
        raise RuntimeError(f"[{chunk_name}] 두 경로 벡터가 다름(min {min(sims):.4f}) → 재사용 불가, 설정 확인")
run_meta["path_equivalence"] = equiv

[A_fixed] 두 경로 코사인 유사도 최소 1.000000 / 평균 1.000000
[B_structure] 두 경로 코사인 유사도 최소 1.000000 / 평균 1.000000


## 5. 지역 필터 + 필터 검증 규칙
**지역 필터 (모든 모델에 똑같이)**
- 시도: 내 시도 **또는** 시도가 비어 있음 → PDF(전국 지침)는 `sido_name = null`이라 항상 포함
- 시군구: 내 시군구 **또는** 시군구가 비어 있음 → 시도 전체 정책도 포함
- `IsEmpty` = 값이 null이거나 칸 자체가 없는 경우

**필터 검증 규칙 (qid별)** — 유사도와 무관하게 "필터만" 검사하기 위한 허용/차단 ID
- smoke: `SMOKE_QIDS` 6개 / **본 실행: 채점 질문 36개 전부 + V37** (지역짝 V19~V24의 `must_not_ids`가 모두 검사됨)
- `allow`: 그 질문의 정답 → 필터를 통과해야 함
- `block`: 그 질문의 `must_not_ids` + `SMOKE_BLOCK_FROM`에 적힌 다른 질문의 정답 → 통과하면 안 됨
- `block`은 **그 qid에서만** 차단 대상이다 (예: V20 정답은 V19에서만 차단)

In [9]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, IsEmptyCondition, PayloadField

def match_or_empty(field, value):
    key = f"metadata.{field}"
    return Filter(should=[
        FieldCondition(key=key, match=MatchValue(value=value)),
        IsEmptyCondition(is_empty=PayloadField(key=key)),
    ])

def region_filter(q):
    if not USE_REGION_FILTER or not q.get("user_sido"):
        return None
    conditions = [match_or_empty("sido_name", q["user_sido"])]
    if q.get("user_sigungu"):
        conditions.append(match_or_empty("sigungu_name", q["user_sigungu"]))
    return Filter(must=conditions)

def build_filter_rules(qids):
    rules = {}
    for qid in qids:
        q = q_by_id[qid]
        allow = set(q["expected_ids"])
        block = set(q.get("must_not_ids") or [])
        for src in SMOKE_BLOCK_FROM.get(qid, []):
            block |= set(q_by_id[src]["expected_ids"])
        rules[qid] = {"allow": allow, "block": block}
    return rules

FILTER_RULES_SMOKE = build_filter_rules(SMOKE_QIDS)
FILTER_RULES_FULL = build_filter_rules(SCORING_QIDS + [q["qid"] for q in trap_questions if q.get("must_not_ids")])   # 36 + V37
FILTER_RULES = FILTER_RULES_SMOKE if SMOKE else FILTER_RULES_FULL
print(f"필터 규칙: smoke {len(FILTER_RULES_SMOKE)}개 / 본 실행 {len(FILTER_RULES_FULL)}개 "
      f"(차단 규칙 있는 질문 {sum(1 for r in FILTER_RULES_FULL.values() if r['block'])}개) → 이번 실행 {len(FILTER_RULES)}개")
for qid, r in FILTER_RULES_SMOKE.items():
    print(qid, q_by_id[qid]["user_sido"], q_by_id[qid].get("user_sigungu"), "| allow", sorted(r["allow"]), "| block", sorted(r["block"]))

필터 규칙: smoke 6개 / 본 실행 116개 (차단 규칙 있는 질문 24개) → 이번 실행 116개
V01 서울특별시 마포구 | allow ['local:WLF00000696'] | block ['local:WLF00002700']
V13 전라남도 목포시 | allow ['local:WLF00003571'] | block ['local:WLF00005113', 'local:WLF00005491']
V19 경기도 동두천시 | allow ['local:WLF00005912'] | block ['local:WLF00006344']
V25 서울특별시 성북구 | allow ['local:WLF00002231'] | block []
V27 None None | allow ['basic_pension_2026_p0021_03'] | block []
V37 경기도 광명시 | allow [] | block ['local:WLF00005912', 'local:WLF00006344']


## 6. 평가 함수
- **문서 벡터 → Qdrant 직접 적재 → `QdrantVectorStore`로 감싸기**. 두 모델 모두 같은 적재 코드를 쓴다. Qwen은 4-2에서 보관한 벡터를, KURE는 LangChain `embed_documents`로 만든 벡터를 쓴다(본 실행 때 만든 전체 벡터는 같은 커널 안에서 재사용)
- payload 구조(`page_content`, `metadata`)·COSINE·차원은 `QdrantVectorStore.from_documents`와 동일. point id는 `uuid5(chunk_id)`
- **메모리(`:memory:`)에만** 적재 → Docker Qdrant 컬렉션은 건드리지 않음
- 시간은 **문서 임베딩(초) / 적재(초, upsert만) / 검색(초/질문)** 으로 나눠 기록. 재사용 벡터의 문서 임베딩 시간은 4-2에서 잰 전체 세트 값이다. smoke에서는 KURE가 서브셋만 임베딩하므로 두 모델의 임베딩 시간을 비교하지 않는다(본 실행에서만 비교)
- **필터 검증**: 적재 직후, 평가에 쓰는 것과 **같은 `Filter` 객체**를 `scroll`에 넘겨 통과 parent_id를 **페이지 끝까지** 모은다. 규칙 ID가 컬렉션에 없거나 `allow ⊄ 통과` 또는 `block ∩ 통과 ≠ ∅`면 **중단**
- 정답 판정: 결과 청크의 `parent_id ∈ expected_ids`. 같은 부모의 청크가 여러 개 나와도 첫 순위만
- smoke 서브셋: `SMOKE_QIDS`의 allow·block 청크 전부 + 무작위 `SMOKE_RANDOM`개 (seed 고정, 청킹 세트별 1회 생성 → 두 모델에 동일 목록)

In [10]:
import random
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from langchain_qdrant import QdrantVectorStore

COLLECTION = "ab_test"

def build_smoke_subset(docs, rules):
    want = set()
    for r in rules.values():
        want |= r["allow"] | r["block"]
    fixed = [d for d in docs if d.metadata["parent_id"] in want]
    rest = [d for d in docs if d.metadata["parent_id"] not in want]
    sample = random.Random(SEED).sample(rest, SMOKE_RANDOM)
    return fixed + sample

def get_doc_vectors(model_name, embedding_model, chunk_name, docs):
    # 보관된 전체 세트 벡터가 있으면 재사용, 없으면 LangChain 경로로 생성 (전체 세트면 보관)
    cache = doc_vectors[chunk_name].get(model_name)
    full_set = len(docs) == len(chunk_data[chunk_name])
    if cache is None and full_set:                                # 디스크 캐시 확인 (KURE 본 실행 재사용)
        cache = load_vectors(cache_path(model_name, chunk_name), chunk_data[chunk_name], model_name)
        if cache is not None:
            doc_vectors[chunk_name][model_name] = cache
    if cache is not None:
        return [cache["by_chunk_id"][d.metadata["chunk_id"]] for d in docs], cache["embed_sec"], cache["source"]
    t0 = time.time()
    vecs = embedding_model.embed_documents([d.page_content for d in docs])
    sec = round(time.time() - t0, 1)
    check_vectors(vecs, len(docs), dims[model_name], f"{chunk_name}/{model_name}")
    if full_set:
        source = "본 실행에서 LangChain embed_documents로 생성"
        doc_vectors[chunk_name][model_name] = {"by_chunk_id": {d.metadata["chunk_id"]: v for d, v in zip(docs, vecs)},
                                               "embed_sec": sec, "source": source}
        save_vectors(cache_path(model_name, chunk_name), docs, vecs, sec, source, model_name, {"langchain_embed_documents": True})
        return vecs, sec, source
    return vecs, sec, f"서브셋 {len(docs)}개 LangChain embed_documents로 생성(미보관)"

def build_vectorstore(docs, vecs, dim, embedding_model):
    if len(vecs) != len(docs):
        raise RuntimeError(f"벡터 수 {len(vecs)} != 청크 수 {len(docs)}")
    client = QdrantClient(location=":memory:")
    client.create_collection(COLLECTION, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
    vectorstore = QdrantVectorStore(client=client, collection_name=COLLECTION, embedding=embedding_model)  # 생성자가 작은 embed 호출을 하므로 적재 시간에서 제외
    points = [PointStruct(id=str(uuid.uuid5(uuid.NAMESPACE_URL, d.metadata["chunk_id"])), vector=v,
                          payload={QdrantVectorStore.CONTENT_KEY: d.page_content, QdrantVectorStore.METADATA_KEY: d.metadata})
              for d, v in zip(docs, vecs)]
    t0 = time.time()
    for i in range(0, len(points), 256):
        client.upsert(COLLECTION, points[i:i+256])
    upsert_sec = time.time() - t0
    if client.count(COLLECTION).count != len(docs):
        raise RuntimeError("적재 개수 불일치")
    return vectorstore, upsert_sec

def passed_parent_ids(client, collection, flt):
    ids, offset = set(), None
    while True:
        points, offset = client.scroll(collection_name=collection, scroll_filter=flt, limit=1000,
                                       offset=offset, with_payload=True, with_vectors=False)
        ids.update(p.payload["metadata"]["parent_id"] for p in points)
        if offset is None:
            return ids

def check_filters(vectorstore, rules, label):
    client, coll = vectorstore.client, vectorstore.collection_name
    all_ids = passed_parent_ids(client, coll, None)
    rows = []
    for qid, r in rules.items():
        missing = (r["allow"] | r["block"]) - all_ids
        if missing:
            raise RuntimeError(f"[{label}] {qid} 규칙 ID가 컬렉션에 없음 → 검사 무의미: {sorted(missing)}")
        flt = region_filter(q_by_id[qid])
        if flt is None:
            rows.append({"label": label, "qid": qid, "checked": False, "passed_count": len(all_ids), "ok": None,
                         "note": "지역 지정 없음 → 전체 통과, 검증 제외"})
            continue
        passed = passed_parent_ids(client, coll, flt)
        ok = (r["allow"] <= passed) and not (r["block"] & passed)
        rows.append({"label": label, "qid": qid, "checked": True, "passed_count": len(passed), "ok": ok,
                     "allow_missing": sorted(r["allow"] - passed), "block_leaked": sorted(r["block"] & passed), "note": ""})
        if not ok:
            raise RuntimeError(f"[{label}] {qid} 필터 검증 실패: allow 누락 {sorted(r['allow'] - passed)}, block 통과 {sorted(r['block'] & passed)}")
    print(f"[{label}] 필터 검증 통과 ({sum(1 for x in rows if x['checked'])}개 qid 검사)")
    return rows

def search(vectorstore, q):
    return vectorstore.similarity_search_with_score(q["question"], k=K, filter=region_filter(q))

def evaluate(model_name, embedding_model, chunk_name, docs, questions, traps, rules):
    label = f"{chunk_name}/{model_name}"
    dim = len(embedding_model.embed_query("준비"))            # 워밍업

    vecs, embed_sec, source = get_doc_vectors(model_name, embedding_model, chunk_name, docs)
    vectorstore, build_sec = build_vectorstore(docs, vecs, dim, embedding_model)   # 적재(초) = upsert만

    filter_rows = check_filters(vectorstore, rules, label)   # 실패 시 raise

    hits = {1: 0, 3: 0, 5: 0}
    rr_sum = recall_sum = 0
    details = []
    t1 = time.time()
    for q in questions:
        results = search(vectorstore, q)
        expected = set(q["expected_ids"])
        block = rules.get(q["qid"], {}).get("block", set())
        rank, found, blocked_hits = None, set(), 0
        for i, (doc, score) in enumerate(results, start=1):
            pid = doc.metadata.get("parent_id")
            if pid in expected:
                found.add(pid)
                if rank is None:
                    rank = i
            if pid in block:
                blocked_hits += 1
        for n in hits:
            if rank is not None and rank <= n:
                hits[n] += 1
        if rank is not None:
            rr_sum += 1 / rank
        recall_sum += len(found) / len(expected)
        first_doc, first_score = results[0] if results else (None, None)
        details.append({
            "chunk_set": chunk_name, "model": model_name, "qid": q["qid"], "q_type": q["q_type"],
            "question": q["question"], "rank": rank, "recall": round(len(found) / len(expected), 2),
            "n_returned": len(results), "blocked_in_topk": blocked_hits,
            "top1_score": round(first_score, 3) if first_doc else None,
            "top1_title": first_doc.metadata.get("title") if first_doc else None,
        })
    search_sec = time.time() - t1                             # 질문 반복 직후 종료

    trap_rows = []
    for q in traps:
        results = search(vectorstore, q)
        pids = [d.metadata.get("parent_id") for d, _ in results]
        sidos = [d.metadata.get("sido_name") for d, _ in results]
        must_not = set(q.get("must_not_ids") or [])
        trap_rows.append({
            "chunk_set": chunk_name, "model": model_name, "qid": q["qid"], "q_type": q["q_type"],
            "question": q["question"], "n_returned": len(results),
            "record_types": dict(Counter(d.metadata.get("record_type") for d, _ in results)),
            "sido_values": sidos, "must_not_hits": sum(p in must_not for p in pids),
            "local_policy_hits": sum(s is not None for s in sidos),
            "top1_score": round(results[0][1], 3) if results else None,
            "top1_title": results[0][0].metadata.get("title") if results else None,
            "empty": len(results) == 0,
        })

    total = len(questions)
    summary = {
        "chunk_set": chunk_name, "model": model_name, "mode": "smoke" if SMOKE else "full",
        "n_chunks": len(docs), "n_questions": total, "dim": dim,
        "Hit@1": round(hits[1] / total, 3), "Hit@3": round(hits[3] / total, 3), "Hit@5": round(hits[5] / total, 3),
        "Recall@5": round(recall_sum / total, 3), "MRR@5": round(rr_sum / total, 3),
        "문서임베딩(초)": embed_sec, "적재(초)": round(build_sec, 1), "검색(초/질문)": round(search_sec / total, 2),
        "벡터출처": source,
    }
    return summary, details, trap_rows, filter_rows

## 7. 전체 실행 + 결과표 (청킹 방식별, MRR@5 높은 순)
- `SMOKE=True`: 질문 6개(함정 V37 포함) · 소규모 청크로 두 구성 × 두 청킹 실행 → 코드·필터·토큰 경로 점검. **점수는 의미 없음**
- `SMOKE=False`: 36개 질문 + 함정 4개, 전체 청크. Qwen 벡터는 4-2에서 재사용, KURE 벡터는 여기서 처음 만들고 같은 커널 안에서 재사용. ⏱ KURE 전체 임베딩은 CPU에서 수십 분, GPU(RTX 4090 실측)에서는 청킹당 약 25초
- smoke 뒤 본 실행으로 바꿀 때는 **설정 셀에서 `SMOKE = False`로 바꾸고 설정 셀부터 다시 실행**한다(`RUN_ID`가 새로 생김). 설정 셀부터 다시 돌리면 메모리의 `doc_vectors`는 초기화되므로 실제 재사용은 **디스크 캐시**(`results/cache/`)로 이뤄진다. 4-2에서 Qwen 캐시를 로드하고, KURE는 본 실행에서 처음 만들어 캐시에 저장한다
- 조합(청킹 × 구성) 하나가 끝날 때마다 `results/{RUN_ID}/partial_*.csv`로 중간 저장한다(중간에 끊겨도 완료된 조합은 남음)

In [11]:
import os
os.makedirs(RESULT_DIR, exist_ok=False)                       # 실행 폴더 선생성 (덮어쓰기 불가)
summaries, all_details, all_traps, all_filter_rows = [], [], [], []
smoke_chunk_ids = {}

def save_partial():
    pd.DataFrame(summaries).to_csv(f"{RESULT_DIR}/partial_summary.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_details).to_csv(f"{RESULT_DIR}/partial_detail.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_traps).to_csv(f"{RESULT_DIR}/partial_trap.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_filter_rows).to_csv(f"{RESULT_DIR}/partial_filter_check.csv", index=False, encoding="utf-8-sig")

for chunk_name in CHUNK_SETS:
    if SMOKE:
        docs = build_smoke_subset(chunk_data[chunk_name], FILTER_RULES)
        smoke_chunk_ids[chunk_name] = [d.metadata["chunk_id"] for d in docs]
        questions = [q for q in chunk_questions[chunk_name] if q["qid"] in SMOKE_QIDS]
        traps = [q for q in trap_questions if q["qid"] in SMOKE_QIDS]
    else:
        docs = chunk_data[chunk_name]
        questions = chunk_questions[chunk_name]
        traps = trap_questions
    print(f"\n=== [{chunk_name}] 청크 {len(docs)}개, 채점 질문 {len(questions)}개, 함정 {len(traps)}개 ===")
    for model_name, embedding_model in models.items():
        print(f"평가 중: [{chunk_name}] {model_name}")
        summary, details, trap_rows, filter_rows = evaluate(model_name, embedding_model, chunk_name,
                                                            docs, questions, traps, FILTER_RULES)
        summaries.append(summary); all_details.extend(details)
        all_traps.extend(trap_rows); all_filter_rows.extend(filter_rows)
        save_partial()
        print("  ", {k: v for k, v in summary.items() if k not in ("chunk_set", "model", "mode", "벡터출처")})

result_df = pd.DataFrame(summaries).sort_values(["chunk_set", "MRR@5"], ascending=[True, False])
result_df


=== [A_fixed] 청크 3400개, 채점 질문 113개, 함정 9개 ===
평가 중: [A_fixed] qwen3-embedding:0.6b
[A_fixed/qwen3-embedding:0.6b] 필터 검증 통과 (59개 qid 검사)
   {'n_chunks': 3400, 'n_questions': 113, 'dim': 1024, 'Hit@1': 0.487, 'Hit@3': 0.735, 'Hit@5': 0.77, 'Recall@5': 0.722, 'MRR@5': 0.606, '문서임베딩(초)': 34.3, '적재(초)': 0.5, '검색(초/질문)': 0.38}
평가 중: [A_fixed] KURE-v1
[A_fixed/KURE-v1] 필터 검증 통과 (59개 qid 검사)
   {'n_chunks': 3400, 'n_questions': 113, 'dim': 1024, 'Hit@1': 0.69, 'Hit@3': 0.841, 'Hit@5': 0.885, 'Recall@5': 0.85, 'MRR@5': 0.765, '문서임베딩(초)': 24.2, '적재(초)': 0.5, '검색(초/질문)': 0.24}

=== [B_structure] 청크 3865개, 채점 질문 113개, 함정 9개 ===
평가 중: [B_structure] qwen3-embedding:0.6b
[B_structure/qwen3-embedding:0.6b] 필터 검증 통과 (59개 qid 검사)
   {'n_chunks': 3865, 'n_questions': 113, 'dim': 1024, 'Hit@1': 0.487, 'Hit@3': 0.699, 'Hit@5': 0.779, 'Recall@5': 0.732, 'MRR@5': 0.596, '문서임베딩(초)': 36.9, '적재(초)': 0.5, '검색(초/질문)': 0.4}
평가 중: [B_structure] KURE-v1
[B_structure/KURE-v1] 필터 검증 통과 (59개 qid 검사)
   {'n_chunks': 38

,chunk_set,model,mode,n_chunks,n_questions,dim,Hit@1,Hit@3,Hit@5,Recall@5,MRR@5,문서임베딩(초),적재(초),검색(초/질문),벡터출처
1,A_fixed,KURE-v1,full,3400,113,1024,0.690,0.841,0.885,0.850,0.765,24.2,0.5,0.24,"디스크 캐시(KURE-v1__A_fixed__4a35133a270bc941.npz,..."
0,A_fixed,qwen3-embedding:0.6b,full,3400,113,1024,0.487,0.735,0.770,0.722,0.606,34.3,0.5,0.38,디스크 캐시(qwen3-embedding_0.6b__A_fixed__e2210d50...
3,B_structure,KURE-v1,full,3865,113,1024,0.619,0.796,0.858,0.816,0.713,23.1,0.6,0.25,디스크 캐시(KURE-v1__B_structure__6f32006310702240....
2,B_structure,qwen3-embedding:0.6b,full,3865,113,1024,0.487,0.699,0.779,0.732,0.596,36.9,0.5,0.40,디스크 캐시(qwen3-embedding_0.6b__B_structure__a3d7...


## 8. 질문 유형별 정답률 · 함정 점검 · 필터 검증 결과
- 유형별: 지역정책 · 지역짝 · 시도전체 · PDF근거별로 5등 안에 정답이 나온 비율
- 함정 점검 기대값: V37 `must_not_hits = 0` / V38·V39 `local_policy_hits = 0`이고 반환 문서는 PDF만 / 빈 결과(`empty = True`)는 기대값 충족이 아니라 "비어 있음"으로 따로 본다 / V40은 Top-1 점수·제목 관찰만(임계값 판단 없음)

In [12]:
detail_df = pd.DataFrame(all_details)
detail_df["hit"] = detail_df["rank"].notna()
display(detail_df.pivot_table(index=["chunk_set", "model"], columns="q_type", values="hit", aggfunc="mean").round(2))

trap_df = pd.DataFrame(all_traps)
display(trap_df[["chunk_set", "model", "qid", "q_type", "n_returned", "empty", "record_types", "sido_values",
                 "must_not_hits", "local_policy_hits", "top1_score", "top1_title"]])

filter_df = pd.DataFrame(all_filter_rows)
display(filter_df)

q_type                            PDF근거  시도전체  지역정책  지역짝
chunk_set   model                                       
A_fixed     KURE-v1                0.82  0.67  1.00  0.9
            qwen3-embedding:0.6b   0.67  0.67  0.95  0.7
B_structure KURE-v1                0.79  0.67  1.00  0.8
            qwen3-embedding:0.6b   0.68  0.67  0.95  0.7

,chunk_set,model,qid,q_type,n_returned,empty,record_types,sido_values,must_not_hits,local_policy_hits,top1_score,top1_title
0,A_fixed,qwen3-embedding:0.6b,V37,함정_지역오추천,5,False,{'guideline': 5},"[None, None, None, None, None]",0,0,0.480,2026년 노인보건복지사업안내 2권
1,A_fixed,qwen3-embedding:0.6b,V38,함정_데이터없는지역,5,False,{'guideline': 5},"[None, None, None, None, None]",0,0,0.528,2026 나에게 힘이 되는 복지서비스
2,A_fixed,qwen3-embedding:0.6b,V39,함정_데이터없는지역,5,False,{'guideline': 5},"[None, None, None, None, None]",0,0,0.510,2026년 노인보건복지사업안내 2권
3,A_fixed,qwen3-embedding:0.6b,V40,함정_범위밖,5,False,{'guideline': 5},"[None, None, None, None, None]",0,0,0.306,2026 나에게 힘이 되는 복지서비스
4,A_fixed,qwen3-embedding:0.6b,W69,함정_지역오추천,5,False,"{'guideline': 4, 'policy': 1}","[None, None, None, 전라남도, None]",0,1,0.460,2026 나에게 힘이 되는 복지서비스
5,A_fixed,qwen3-embedding:0.6b,W70,함정_지역오추천,5,False,"{'policy': 4, 'guideline': 1}","[서울특별시, None, 서울특별시, 서울특별시, 서울특별시]",0,4,0.576,장애인 휠체어 보장구 수리비 지원
6,A_fixed,qwen3-embedding:0.6b,W71,함정_데이터없는지역,5,False,{'guideline': 5},"[None, None, None, None, None]",0,0,0.520,2026년 노인보건복지사업안내 1권
7,A_fixed,qwen3-embedding:0.6b,W72,함정_범위밖,5,False,{'guideline': 5},"[None, None, None, None, None]",0,0,0.443,2026년 노인보건복지사업안내 1권
8,A_fixed,qwen3-embedding:0.6b,T10,함정_시설데이터,5,False,"{'policy': 3, 'guideline': 2}","[전라남도, 전라남도, None, 전라남도, None]",0,3,0.519,골목호랑이 어르신단 운영
9,A_fixed,KURE-v1,V37,함정_지역오추천,5,False,{'guideline': 5},"[None, None, None, None, None]",0,0,0.571,2026 나에게 힘이 되는 복지서비스


,label,qid,checked,passed_count,ok,note,allow_missing,block_leaked
0,A_fixed/qwen3-embedding:0.6b,T01,False,1763,None,"지역 지정 없음 → 전체 통과, 검증 제외",NaN,NaN
1,A_fixed/qwen3-embedding:0.6b,T02,False,1763,None,"지역 지정 없음 → 전체 통과, 검증 제외",NaN,NaN
2,A_fixed/qwen3-embedding:0.6b,T03,False,1763,None,"지역 지정 없음 → 전체 통과, 검증 제외",NaN,NaN
3,A_fixed/qwen3-embedding:0.6b,T04,False,1763,None,"지역 지정 없음 → 전체 통과, 검증 제외",NaN,NaN
4,A_fixed/qwen3-embedding:0.6b,T05,False,1763,None,"지역 지정 없음 → 전체 통과, 검증 제외",NaN,NaN
...,...,...,...,...,...,...,...,...
459,B_structure/KURE-v1,W67,True,1200,True,,[],[]
460,B_structure/KURE-v1,W68,True,1194,True,,[],[]
461,B_structure/KURE-v1,V37,True,1224,True,,[],[]
462,B_structure/KURE-v1,W69,True,1196,True,,[],[]


## 9. 틀린 질문 보기 + 결과 저장
- 틀린 질문을 보면 구성의 약점을 알 수 있다
- 저장 위치 `results/{RUN_ID}/` (7번에서 `exist_ok=False`로 생성 → 덮어쓰기 불가). CSV는 `utf-8-sig`(엑셀 한글 안 깨짐). 7번의 `partial_*.csv`는 중간 저장본
- `run_meta.json`에 환경·해시·채점 qid·smoke 청크 ID를 함께 저장

In [13]:
display(detail_df[~detail_df["hit"]][["chunk_set", "model", "qid", "q_type", "question", "top1_title"]])

result_df.to_csv(f"{RESULT_DIR}/summary.csv", index=False, encoding="utf-8-sig")
detail_df.to_csv(f"{RESULT_DIR}/detail.csv", index=False, encoding="utf-8-sig")
trap_df.to_csv(f"{RESULT_DIR}/trap.csv", index=False, encoding="utf-8-sig")
filter_df.to_csv(f"{RESULT_DIR}/filter_check.csv", index=False, encoding="utf-8-sig")
run_meta["smoke_chunk_ids"] = smoke_chunk_ids
run_meta["filter_rules"] = {qid: {k: sorted(v) for k, v in r.items()} for qid, r in FILTER_RULES.items()}
with open(f"{RESULT_DIR}/run_meta.json", "w", encoding="utf-8") as f:
    json.dump(run_meta, f, ensure_ascii=False, indent=1)
print("저장 완료:", RESULT_DIR)

,chunk_set,model,qid,q_type,question,top1_title
13,A_fixed,qwen3-embedding:0.6b,V14,지역정책,화순에 혼자 사는데 반찬 갖다주면서 들여다봐 주는 거 있어?,화순 사평빨래방 운영
18,A_fixed,qwen3-embedding:0.6b,V19,지역짝,병원 갈 때 같이 가 줄 사람이 없어,2026 나에게 힘이 되는 복지서비스
19,A_fixed,qwen3-embedding:0.6b,V20,지역짝,병원 갈 때 같이 가 줄 사람이 없어,2026 나에게 힘이 되는 복지서비스
21,A_fixed,qwen3-embedding:0.6b,V22,지역짝,목욕비 좀 지원받을 수 있어?,2026년 노인보건복지사업안내 2권
25,A_fixed,qwen3-embedding:0.6b,V26,시도전체,수원 사는데 화장실에 손잡이 달고 문턱 없애주는 거 해주는 데 있어?,2026년 노인보건복지사업안내 2권
...,...,...,...,...,...,...
443,B_structure,KURE-v1,T01,PDF근거,엄마가 올해 예순다섯인데요. 매달 생활비 보태주는 게 있다고 들었는데 무슨 제도예요?,자녀출산 무주택가구 주거비 지원
444,B_structure,KURE-v1,T02,PDF근거,기초연금 신청하러 갈 때 뭐 챙겨가야 돼요?,2026년 기초연금 사업안내
445,B_structure,KURE-v1,T03,PDF근거,국민연금 받고 있는데 기초연금도 신청할 수 있어요?,2026년 기초연금 사업안내
446,B_structure,KURE-v1,T04,PDF근거,아버지가 혼자 사세요. 가끔 집에 와서 안부 봐주는 서비스도 있나요?,2026 나에게 힘이 되는 복지서비스


저장 완료: ./results/20261110_062652_1d153850


## 10. 결과 정리 (2026-11-10 본 실행, Runpod · NVIDIA RTX 4090)
| 항목 | 내용 |
|---|---|
| 실행 기록 | smoke `results/20261110_034520_5cde7f40` (청킹별 212청크·채점 5문항 + 함정 1문항) → 본 실행 `results/20261110_035255_fe62ec73` (전체 청크·채점 36문항 + 함정 4문항). 환경·버전·digest는 각 `run_meta.json` |
| 비교한 구성 | qwen3-embedding:0.6b (Ollama 0.40.16, Q8_0, num_ctx 2048, truncate=False, GPU) vs KURE-v1 (fp32, cuda, revision `8b418a5…`, 정규화) — **지시문 없음 조건** |
| 비교한 청킹 방식 | A_fixed (3,400청크) / B_structure (3,865청크) |
| 선택 구성 | **KURE-v1** |
| 벡터 길이 | 1024 (두 구성 동일) |
| 선택한 청킹 방식 | 미정. KURE-v1 기준 A_fixed(MRR@5 0.765·Hit@1 0.694) vs B_structure(0.759·0.667), Hit@5·Recall@5는 동일(0.889·0.861)이고 MRR 차이가 작아 선택 보류. 청크 데이터 확정 단계에서 별도 결정 |
| 선택 이유 | ① **정확도**: A_fixed에서 KURE-v1 Hit@5 0.889 / MRR@5 0.765 / Recall@5 0.861 vs Qwen 0.750 / 0.590 / 0.736, B_structure도 같은 방향(KURE 0.889 / 0.759 / 0.861 vs Qwen 0.750 / 0.566 / 0.736). 두 청킹 모두 KURE 우세. ② **지역 오류**: 두 구성 모두 함정 4문항 must_not 0건·지역정책 유입 0건, 지역 필터 검증 108/108 통과(네 조합 합계), 지역짝 차단 청크 Top-5 유입 0건 → 차이 없음. ③ 속도·운영: 정확도 차이가 커서 판단에 쓰지 않음. GPU 측정값(문서 임베딩 KURE 23.1~24.2초 / Qwen 34.3~36.9초, 검색 0.09~0.13초/질문)은 기록만 하고 운영 환경 CPU 벤치마크는 별도 |
| 약점 | KURE-v1 Top-5 실패: A_fixed V23·V26·V27·V31, B_structure V23·V26·V31·V36 — 유형별 Hit@5: 지역정책 1.00, 지역짝 0.83, PDF근거 0.80, 시도전체 0.50(2문항 중 1개). Qwen은 9문항 실패(V14·V19·V20·V22·V26·V27·V30·V31·V36), 특히 지역짝 0.50·PDF근거 0.60 |
| 검증 | 잘림 없음(최대 토큰 Qwen 731 / KURE 606 < 2048, 질문 최대 31), 4-3 표본(청킹별 20개) 두 경로 코사인 1.0, 실패 기록 없음 |
| 한계 | 구성 비교이지 모델 구조 비교가 아님(Qwen은 지시문·양자화 조건이 바뀌면 결과가 달라질 수 있음) · 채점 36문항이라 표본이 작음 · 지역 필터 전제(빈 시군구 = 시도 전체)는 사람이 확인한 데이터 설계 전제 · 지역짝 성능은 필터 적용 후 성능 · 속도는 GPU 값이라 운영 판단에 그대로 쓰지 않음 · 함정 16행 모두 결과 5개(지침 문서)를 반환했으므로 지역정책 유입은 없지만 무관 질문 거절·무응답 능력은 검증하지 않음 |

→ 다음: 결과 공유 → 청킹 방식(A/B) 확정 → KURE-v1로 본 임베딩 → Qdrant 적재. 질문 임베딩과 Neo4j도 같은 구성(KURE-v1, 정규화, 지시문 없음)을 쓴다.
